# Banco Multiagente - CP4 - Prompt & IA

### Instalação e configuração


In [ ]:
%pip install -q -U openai-agents

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.9/357.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 45.2 MB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError("Adicione OPENAI_API_KEY aos Secrets do Google Colab.")

os.environ["OPENAI_API_KEY"] = api_key

### Dados fictícios do banco

In [ ]:
# gerado aleatoriamente pela IA
conta_corrente = {
    "titular": "João Silva",
    "agencia": "0001",
    "numero": "12345-6",
    "saldo": 2_850.00,
}

contatos_pix: list[dict] = [
    {"nome": "Ana Souza",     "chave": "ana.souza@email.com",    "tipo_chave": "e-mail"},
    {"nome": "Bruno Lima",    "chave": "11987654321",            "tipo_chave": "celular"},
    {"nome": "Carla Mendes",  "chave": "carla.mendes@email.com", "tipo_chave": "e-mail"},
    {"nome": "Diego Costa",   "chave": "22912345678",            "tipo_chave": "celular"},
    {"nome": "Eva Rocha",     "chave": "111.222.333-44",         "tipo_chave": "CPF"},
]

historico_pix: list[dict] = []

print(f"Titular: {conta_corrente['titular']}")
print(f"Saldo inicial: R$ {conta_corrente['saldo']:,.2f}")
print(f"Contatos cadastrados: {len(contatos_pix)}")


Titular: João Silva
Saldo inicial: R$ 2,850.00
Contatos cadastrados: 5


### Tools do Agente PIX

In [ ]:
from agents import function_tool


@function_tool
def consultar_saldo() -> dict:
    """
    Retorna o saldo atual da conta corrente do cliente logado.
    Use esta ferramenta sempre que o cliente perguntar sobre o saldo disponível
    ou antes de realizar um PIX.
    """
    return {
        "titular": conta_corrente["titular"],
        "agencia": conta_corrente["agencia"],
        "conta": conta_corrente["numero"],
        "saldo_disponivel": conta_corrente["saldo"],
    }


@function_tool
def buscar_contato(nome: str) -> dict:
    """
    Busca um destinatário na lista de contatos PIX cadastrados pelo cliente.
    Faz busca parcial e sem distinção de maiúsculas/minúsculas.
    Retorna os dados do contato encontrado ou uma mensagem de erro.

    Args:
        nome: Nome (ou parte do nome) do destinatário a localizar.
    """
    nome_lower = nome.lower().strip()
    encontrados = [
        c for c in contatos_pix
        if nome_lower in c["nome"].lower()
    ]

    if not encontrados:
        return {"erro": f"Nenhum contato encontrado para '{nome}'. Verifique o nome ou cadastre a chave manualmente."}

    if len(encontrados) > 1:
        return {
            "aviso": "Mais de um contato encontrado. Peça ao usuário para confirmar qual deseja.",
            "contatos": encontrados,
        }

    return {"contato": encontrados[0]}


@function_tool
def realizar_pix(nome_destinatario: str, chave_pix: str, valor: float, mensagem: str = "") -> dict:
    """
    Executa um PIX simulado, debitando o valor da conta do cliente.
    Deve ser chamada APENAS após o usuário confirmar os dados da transação.
    Valida saldo suficiente antes de prosseguir.

    Args:
        nome_destinatario: Nome completo do destinatário.
        chave_pix: Chave PIX do destinatário (e-mail, CPF ou celular).
        valor: Valor em reais a transferir (deve ser positivo).
        mensagem: Descrição opcional da transferência.
    """
    if valor <= 0:
        return {"erro": "O valor do PIX deve ser maior que zero."}

    if valor > conta_corrente["saldo"]:
        return {
            "erro": "Saldo insuficiente.",
            "saldo_disponivel": conta_corrente["saldo"],
            "valor_solicitado": valor,
        }

    conta_corrente["saldo"] -= valor

    import datetime
    transacao = {
        "id": f"PIX{len(historico_pix) + 1:04d}",
        "destinatario": nome_destinatario,
        "chave": chave_pix,
        "valor": valor,
        "mensagem": mensagem,
        "data_hora": datetime.datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
        "status": "aprovado",
    }
    historico_pix.append(transacao)

    return {
        "sucesso": True,
        "transacao": transacao,
        "novo_saldo": conta_corrente["saldo"],
    }

### Definição dos Agentes

Três agentes:
- **Agente de Dúvidas** — responde sobre produtos e serviços do banco
- **Agente PIX** — opera a conta com as tools definidas acima
- **Agente de Atendimento** — triagem; faz handoff para o especialista correto


In [ ]:
from agents import Agent, Runner

# Dúvidas
agente_duvidas = Agent(
    name="Agente de Dúvidas",
    handoff_description=(
        "Use para perguntas sobre produtos, serviços, tarifas, limites, "
        "cartões, seguros, investimentos, PIX (funcionamento geral) e "
        "regras do banco. NÃO use para operações de conta como consulta de "
        "saldo ou transferências — essas ficam com o Agente PIX."
    ),
    instructions=(
        "Você é o assistente de dúvidas do Banco Fictício S.A. "
        "Responda em português, com clareza e cordialidade. "
        "Use as informações abaixo sobre o banco:\n\n"

        "=== PRODUTOS E SERVIÇOS ===\n"
        "- Conta Corrente: sem tarifa para clientes com renda acima de R$1.500/mês.\n"
        "- Conta Poupança: rendimento de 0,5% a.m. + TR. Sem tarifas.\n"
        "- Cartão de Crédito Básico: limite inicial de R$1.000. Anuidade: R$199/ano (isenta com 1 compra/mês).\n"
        "- Cartão de Crédito Gold: limite inicial de R$5.000. Anuidade: R$49/mês.\n"
        "- Seguro Vida: a partir de R$29/mês. Cobertura de até R$200.000.\n"
        "- CDB: liquidez diária, 100% do CDI. Aplicação mínima: R$500.\n"
        "- Tesouro Direto: disponível pelo app, sem taxa de custódia adicional do banco.\n\n"

        "=== TARIFAS ===\n"
        "- TED: R$8,50 por operação (gratuita via app).\n"
        "- DOC: R$6,00 por operação.\n"
        "- PIX: GRATUITO, 24h por dia, 7 dias por semana.\n"
        "- Saque em caixas do banco: 4 gratuitos/mês; excedentes R$4,50 cada.\n"
        "- Extrato impresso: R$2,00 por via.\n\n"

        "=== PIX — REGRAS ===\n"
        "- Limite diário padrão: R$5.000 (período diurno) e R$1.000 (período noturno, 20h–6h).\n"
        "- Chaves aceitas: CPF, CNPJ, e-mail, celular, chave aleatória.\n"
        "- Prazo de devolução (Pix errado): até 90 dias, mediante solicitação.\n\n"

        "Se não souber a resposta, diga claramente e sugira contato pelo telefone 0800-123-4567."
    ),
    model="gpt-4o-mini",
)

# Pix
agente_pix = Agent(
    name="Agente PIX",
    handoff_description=(
        "Use para operações na conta do cliente: consultar saldo, enviar PIX, "
        "verificar destinatários e confirmar transferências. "
        "NÃO use para dúvidas gerais sobre o banco."
    ),
    instructions=(
        "Você é o agente operacional PIX do Banco Fictício S.A. "
        "Seu papel é executar operações de conta com segurança e transparência.\n\n"

        "FLUXO OBRIGATÓRIO para um PIX:\n"
        "1. Use 'buscar_contato' para identificar o destinatário pelo nome informado.\n"
        "2. Use 'consultar_saldo' para verificar se há saldo suficiente.\n"
        "3. Apresente ao usuário um RESUMO DA TRANSAÇÃO com: destinatário, chave PIX, valor e saldo atual. "
           "Peça confirmação explícita ('Confirma? S/N').\n"
        "4. Somente após confirmação do usuário, chame 'realizar_pix'.\n"
        "5. Informe o comprovante com ID da transação e novo saldo.\n\n"

        "REGRAS:\n"
        "- Nunca execute um PIX sem confirmação explícita do usuário.\n"
        "- Se o contato não for encontrado, explique e ofereça alternativas.\n"
        "- Se o saldo for insuficiente, informe o valor disponível.\n"
        "- Responda sempre em português, com cordialidade e objetividade."
    ),
    tools=[consultar_saldo, buscar_contato, realizar_pix],
    model="gpt-4o-mini",
)

# Atendimento
agente_atendimento = Agent(
    name="Agente de Atendimento",
    instructions=(
        "Você é o recepcionista virtual do Banco Fictício S.A. "
        "Cumprimente o cliente com cordialidade e identifique a intenção da solicitação.\n\n"
        "REGRAS DE ROTEAMENTO:\n"
        "- Solicitações de PIX, transferência, consulta de saldo ou operações de conta "
           "→ handoff para o Agente PIX.\n"
        "- Dúvidas sobre produtos, tarifas, serviços, cartões, investimentos ou regras do banco "
           "→ handoff para o Agente de Dúvidas.\n"
        "- Solicitações ambíguas: peça uma breve clarificação antes de fazer o handoff.\n\n"
        "Seja breve na recepção. O especialista cuidará dos detalhes."
    ),
    handoffs=[agente_pix, agente_duvidas],
    model="gpt-4o-mini",
)

### Função auxiliar de teste

In [ ]:
async def atender(solicitacao: str, historico: list = None) -> tuple[str, list]:
    """
    Executa uma solicitação a partir do Agente de Atendimento.
    Retorna (resposta_final, historico_atualizado).
    """
    msgs = (historico or []) + [{"role": "user", "content": solicitacao}]

    resultado = await Runner.run(agente_atendimento, msgs)

    print(f"Agente que concluiu: {resultado.last_agent.name}")
    print("─" * 60)
    print(resultado.final_output)
    print("─" * 60)

    novo_historico = msgs + [{"role": "assistant", "content": resultado.final_output}]
    return resultado.final_output, novo_historico

### Consulta de saldo


In [ ]:
print("Usuário: 'Qual é meu saldo atual?'\n")
await atender("Qual é meu saldo atual?")

Usuário: 'Qual é meu saldo atual?'



Agente que concluiu: Agente PIX
────────────────────────────────────────────────────────────
Seu saldo atual é de R$ 2.850,00. Como posso ajudá-lo?
────────────────────────────────────────────────────────────


('Seu saldo atual é de R$ 2.850,00. Como posso ajudá-lo?',
 [{'role': 'user', 'content': 'Qual é meu saldo atual?'},
  {'role': 'assistant',
   'content': 'Seu saldo atual é de R$ 2.850,00. Como posso ajudá-lo?'}])

### Dúvida sobre tarifas PIX


In [ ]:
print("Usuário: 'O PIX tem alguma tarifa? E qual o limite diário?'\n")
await atender("O PIX tem alguma tarifa? E qual o limite diário?")


Usuário: 'O PIX tem alguma tarifa? E qual o limite diário?'

Agente que concluiu: Agente de Dúvidas
────────────────────────────────────────────────────────────
O PIX é gratuito e pode ser realizado 24 horas por dia, 7 dias por semana. O limite diário é de R$5.000 durante o período diurno e R$1.000 durante o período noturno (20h às 6h). Se tiver mais alguma dúvida, estou à disposição!
────────────────────────────────────────────────────────────


('O PIX é gratuito e pode ser realizado 24 horas por dia, 7 dias por semana. O limite diário é de R$5.000 durante o período diurno e R$1.000 durante o período noturno (20h às 6h). Se tiver mais alguma dúvida, estou à disposição!',
 [{'role': 'user',
   'content': 'O PIX tem alguma tarifa? E qual o limite diário?'},
  {'role': 'assistant',
   'content': 'O PIX é gratuito e pode ser realizado 24 horas por dia, 7 dias por semana. O limite diário é de R$5.000 durante o período diurno e R$1.000 durante o período noturno (20h às 6h). Se tiver mais alguma dúvida, estou à disposição!'}])

### Contato não encontrado


In [ ]:
print("Usuário: 'Manda R$ 200 para o Fernando'\n")
await atender("Manda R$ 200 para o Fernando")


Usuário: 'Manda R$ 200 para o Fernando'

Agente que concluiu: Agente PIX
────────────────────────────────────────────────────────────
Não encontrei nenhum contato cadastrado com o nome "Fernando". 

Você pode verificar se o nome está correto ou, se preferir, me fornecer a chave PIX (e-mail, CPF ou celular) dele para que eu possa proceder. O que você gostaria de fazer?
────────────────────────────────────────────────────────────


('Não encontrei nenhum contato cadastrado com o nome "Fernando". \n\nVocê pode verificar se o nome está correto ou, se preferir, me fornecer a chave PIX (e-mail, CPF ou celular) dele para que eu possa proceder. O que você gostaria de fazer?',
 [{'role': 'user', 'content': 'Manda R$ 200 para o Fernando'},
  {'role': 'assistant',
   'content': 'Não encontrei nenhum contato cadastrado com o nome "Fernando". \n\nVocê pode verificar se o nome está correto ou, se preferir, me fornecer a chave PIX (e-mail, CPF ou celular) dele para que eu possa proceder. O que você gostaria de fazer?'}])

### Saldo insuficiente

In [ ]:
print("Usuário: 'Preciso transferir R$ 10.000 para o Bruno'\n")
await atender("Preciso transferir R$ 10.000 para o Bruno")


Usuário: 'Preciso transferir R$ 10.000 para o Bruno'

Agente que concluiu: Agente PIX
────────────────────────────────────────────────────────────
Encontrei o contato:

- **Destinatário:** Bruno Lima
- **Chave PIX:** 11987654321 (celular)

### Detalhes da Transação:
- **Valor:** R$ 10.000
- **Saldo Disponível:** R$ 2.850,00

Infelizmente, você não possui saldo suficiente para realizar essa transferência. Seu saldo atual é de R$ 2.850,00.

Se precisar de ajuda para transferir um valor menor ou outra operação, estou à disposição!
────────────────────────────────────────────────────────────


('Encontrei o contato:\n\n- **Destinatário:** Bruno Lima\n- **Chave PIX:** 11987654321 (celular)\n\n### Detalhes da Transação:\n- **Valor:** R$ 10.000\n- **Saldo Disponível:** R$ 2.850,00\n\nInfelizmente, você não possui saldo suficiente para realizar essa transferência. Seu saldo atual é de R$ 2.850,00.\n\nSe precisar de ajuda para transferir um valor menor ou outra operação, estou à disposição!',
 [{'role': 'user', 'content': 'Preciso transferir R$ 10.000 para o Bruno'},
  {'role': 'assistant',
   'content': 'Encontrei o contato:\n\n- **Destinatário:** Bruno Lima\n- **Chave PIX:** 11987654321 (celular)\n\n### Detalhes da Transação:\n- **Valor:** R$ 10.000\n- **Saldo Disponível:** R$ 2.850,00\n\nInfelizmente, você não possui saldo suficiente para realizar essa transferência. Seu saldo atual é de R$ 2.850,00.\n\nSe precisar de ajuda para transferir um valor menor ou outra operação, estou à disposição!'}])

### Dúvida sobre cartão de crédito

In [ ]:
print("Usuário: 'Quais cartões de crédito o banco oferece? Tem anuidade?'\n")
await atender("Quais cartões de crédito o banco oferece? Tem anuidade?")


Usuário: 'Quais cartões de crédito o banco oferece? Tem anuidade?'

Agente que concluiu: Agente de Dúvidas
────────────────────────────────────────────────────────────
O Banco Fictício S.A. oferece os seguintes cartões de crédito:

1. **Cartão de Crédito Básico**:
   - Limite inicial: R$1.000
   - Anuidade: R$199/ano (isenta com 1 compra/mês).

2. **Cartão de Crédito Gold**:
   - Limite inicial: R$5.000
   - Anuidade: R$49/mês.

Se precisar de mais informações, estou à disposição!
────────────────────────────────────────────────────────────


('O Banco Fictício S.A. oferece os seguintes cartões de crédito:\n\n1. **Cartão de Crédito Básico**:\n   - Limite inicial: R$1.000\n   - Anuidade: R$199/ano (isenta com 1 compra/mês).\n\n2. **Cartão de Crédito Gold**:\n   - Limite inicial: R$5.000\n   - Anuidade: R$49/mês.\n\nSe precisar de mais informações, estou à disposição!',
 [{'role': 'user',
   'content': 'Quais cartões de crédito o banco oferece? Tem anuidade?'},
  {'role': 'assistant',
   'content': 'O Banco Fictício S.A. oferece os seguintes cartões de crédito:\n\n1. **Cartão de Crédito Básico**:\n   - Limite inicial: R$1.000\n   - Anuidade: R$199/ano (isenta com 1 compra/mês).\n\n2. **Cartão de Crédito Gold**:\n   - Limite inicial: R$5.000\n   - Anuidade: R$49/mês.\n\nSe precisar de mais informações, estou à disposição!'}])

### Histórico de transações

In [ ]:
print("\nEXTRATO DA SESSÃO")
print("=" * 50)
print(f"Titular: {conta_corrente['titular']}")
print(f"Conta:   {conta_corrente['agencia']} / {conta_corrente['numero']}")
print(f"Saldo:   R$ {conta_corrente['saldo']:,.2f}")
print()

if historico_pix:
    print("Transações PIX realizadas:")
    for t in historico_pix:
        print(f"  [{t['id']}] {t['data_hora']} → {t['destinatario']} | R$ {t['valor']:.2f} | {t['status'].upper()}")
else:
    print("Nenhuma transação realizada nesta sessão.")



EXTRATO DA SESSÃO
Titular: João Silva
Conta:   0001 / 12345-6
Saldo:   R$ 2,850.00

Nenhuma transação realizada nesta sessão.


## Chat interativo

In [ ]:
historico_chat = []

print("Banco — Atendimento Virtual")
print("Digite 'sair' para encerrar.")
print("=" * 50)

while True:
    mensagem = input("\nVocê: ").strip()
    if not mensagem:
        continue
    if mensagem.lower() in ("sair", "exit", "quit"):
        print("\nVocê saiu")
        break

    print()
    _, historico_chat = await atender(mensagem, historico_chat)


Banco — Atendimento Virtual
Digite 'sair' para encerrar.

Você: sair

Você saiu
